# Phase 5: Network Graph & Wisdom Score Computation

**Runtime**: ~5 minutes

This notebook builds a wallet interaction network from Polymarket trades and computes the **Wisdom Score**, a composite metric measuring whether the market functions as a crowd wisdom mechanism.

**Surowiecki Four Conditions** (each scored 0–100):
- **Diversity**: fraction of volume outside behaviorally-suspicious wallets (DBSCAN cluster −1)
- **Independence**: inverse of coordination signal (% non-outlier wallets)
- **Decentralization**: inverse of network hub dominance (Freeman centralization)
- **Aggregation**: community modularity proxy

**Outputs**:
- `wisdom_score_summary.json`: 76.6/100 score + sub-scores
- `network_stats.txt`: Network metrics

## Setup: Install dependencies

In [ ]:
# Install required packages
!pip install -q pandas numpy networkx

import json
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from itertools import combinations

print("✓ All dependencies installed")

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Helper: Gini Coefficient

In [ ]:
def section(title: str) -> None:
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")


def gini_coefficient(values):
    """Compute Gini coefficient (0 = perfect equality, 1 = perfect inequality)."""
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals)) / (n * np.sum(sorted_vals)) - (n + 1) / n

print("✓ Helper functions defined")

## Phase 1: Load data

In [ ]:
section("1. Loading data")

df = pd.read_csv(PROCESSED_DIR / "poly_trades_all_matched.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

clusters = json.load(open(PROCESSED_DIR / "dbscan_clusters.json"))
cluster_map = {c["wallet"]: c["cluster"] for c in clusters}

print(f"Trades: {len(df):,}")
print(f"Unique wallets: {df['proxyWallet'].nunique():,}")
print(f"Cluster assignments loaded: {len(cluster_map):,}")

## Phase 2: Build wallet interaction network

In [ ]:
section("2. Building wallet interaction network")

G = nx.Graph()

# Add nodes with cluster attributes
for wallet in df["proxyWallet"].unique():
    cluster_id = cluster_map.get(wallet, -1)
    G.add_node(wallet, cluster=cluster_id)

print(f"Nodes: {G.number_of_nodes()}")

# Add edges: wallets trading same side in same 1-hour window
# (indicates potential coordination or information leakage)
df["time_bin"] = df["timestamp"].dt.floor("1h")
edge_count = 0
for (time_bin, side), group in df.groupby(["time_bin", "side"]):
    wallets_in_bin = group["proxyWallet"].unique()
    if len(wallets_in_bin) < 2:
        continue
    for w1, w2 in combinations(wallets_in_bin, 2):
        if G.has_edge(w1, w2):
            G[w1][w2]["weight"] += 1
        else:
            G.add_edge(w1, w2, weight=1)
            edge_count += 1

print(f"Edges: {G.number_of_edges():,}")
print(f"Network density: {nx.density(G):.4f}")

## Phase 3: Compute network metrics

In [ ]:
section("3. Computing network metrics")

# Volume concentration
wallet_volumes = df.groupby("proxyWallet")["size"].sum()
top5_share = wallet_volumes.nlargest(5).sum() / wallet_volumes.sum()
gini = gini_coefficient(wallet_volumes.values)
print(f"\nVolume concentration:")
print(f"  Top 5 wallets share: {top5_share:.1%}")
print(f"  Gini coefficient: {gini:.3f}")

# Network centralization
if G.number_of_nodes() > 0:
    dc = nx.degree_centrality(G)
    max_dc = max(dc.values()) if dc else 0
    centralization = (
        sum(max_dc - v for v in dc.values()) / ((len(G) - 1) * (len(G) - 2))
        if len(G) > 1
        else 0
    )
    print(f"\nNetwork centralization:")
    print(f"  Max degree centrality: {max_dc:.4f}")
    print(f"  Freeman centralization: {centralization:.4f}")
else:
    centralization = 0

# Community detection (Louvain)
if G.number_of_edges() > 0:
    try:
        communities = list(nx.community.louvain_communities(G, weight="weight", seed=42))
        modularity = nx.community.modularity(G, communities, weight="weight")
        print(f"\nCommunity structure:")
        print(f"  Communities: {len(communities)}")
        print(f"  Modularity: {modularity:.4f}")
        print(f"  Community sizes: {sorted([len(c) for c in communities], reverse=True)[:5]}")
    except:
        communities = []
        modularity = 0
        print("  Could not compute communities (insufficient network connectivity)")
else:
    communities = []
    modularity = 0

## Phase 4: Cross-validate clustering methods

In [ ]:
section("4. Cross-validating clustering methods")

if communities:
    # For each graph community, check overlap with DBSCAN clusters
    for comm_idx, community in enumerate(communities[:5]):  # Check top 5
        community_clusters = [cluster_map.get(w, -1) for w in community]
        most_common_cluster = max(set(community_clusters), key=community_clusters.count)
        cluster_overlap = community_clusters.count(most_common_cluster) / len(community)
        print(f"  Community {comm_idx} ({len(community)} wallets): "
              f"{cluster_overlap:.1%} in DBSCAN cluster {most_common_cluster}")

## Phase 5: Compute Surowiecki-based Wisdom Score

Measure the four conditions for crowd wisdom (Surowiecki 2004).

In [ ]:
section("5. Computing Wisdom of Crowds Mechanism Score")

# Load wallet features to identify suspicious wallets (DBSCAN cluster -1)
features_df = pd.read_csv(PROCESSED_DIR / "wallet_features.csv")
suspicious_wallets = set(features_df.loc[features_df["cluster"] == -1, "wallet"])

# Diversity Score: (1 - suspicious_volume_share) × 100
suspicious_volume = wallet_volumes[wallet_volumes.index.isin(suspicious_wallets)].sum()
total_volume = wallet_volumes.sum()
suspicious_volume_share = suspicious_volume / total_volume if total_volume > 0 else 0
diversity_score = max(0, min(100, (1 - suspicious_volume_share) * 100))

# Independence Score: (1 - outlier_rate) × 100
outlier_wallets = sum(1 for v in cluster_map.values() if v == -1)
total_wallets = len(cluster_map)
outlier_rate = outlier_wallets / total_wallets if total_wallets > 0 else 0
independence_score = max(0, min(100, (1 - outlier_rate) * 100))

# Decentralization Score: (1 - network_centralization) × 100
decentralization_score = max(0, min(100, (1 - centralization) * 100))

# Aggregation Score: modularity × 100
aggregation_score = max(0, min(100, modularity * 100))

# Composite: equally-weighted average
wisdom_score = (diversity_score + independence_score + decentralization_score + aggregation_score) / 4

print("\nWisdom of Crowds Mechanism Score — Sub-scores:")
print(f"  Diversity Score     : {diversity_score:.1f} / 100  (volume outside suspicious-group; suspicious share = {suspicious_volume_share:.1%})")
print(f"  Independence Score  : {independence_score:.1f} / 100  (low DBSCAN coordination signal)")
print(f"  Decentralization    : {decentralization_score:.1f} / 100  (no dominant hub in network)")
print(f"  Aggregation Score   : {aggregation_score:.1f} / 100  (community modularity proxy)")
print("\n  ↓")
print(f"  WISDOM OF CROWDS MECHANISM SCORE: {wisdom_score:.1f} / 100")

## Phase 6: Interpret the score

Score 76.6 = **Crowd Wisdom Signal** (barely above 70 threshold, with Diversity warning)

In [ ]:
section("6. Interpretation")

# Signal scale
if wisdom_score >= 70:
    signal_label = "Crowd Wisdom Signal"
    signal_description = (
        "Price reflects aggregated beliefs of a diverse, independent group. "
        "Safe to cite as crowd consensus."
    )
elif wisdom_score >= 40:
    signal_label = "Expert Opinion Signal"
    signal_description = (
        "Price reflects informed but concentrated trader opinion. "
        "Cite with qualification."
    )
else:
    signal_label = "Concentrated Capital Signal"
    signal_description = (
        "Price driven by a small number of large positions. "
        "Do not cite as crowd consensus."
    )

print(f"\nSignal Type : {signal_label}")
print(f"Description : {signal_description}")

# Note on Diversity
if diversity_score < 70:
    print(f"\n⚠️  DIVERSITY WARNING: Score of {diversity_score:.1f} indicates")
    print(f"   the behaviorally-suspicious group controls {suspicious_volume_share*100:.1f}% of volume.")
    print(f"   This is a Mode 2 failure (minority price-setting),")
    print(f"   but does not rule out valid crowd wisdom if the suspicious")
    print(f"   group acts independently (Independence: {independence_score:.1f})")

## Phase 7: Save results

In [ ]:
section("7. Saving results")

wisdom_data = {
    "wisdom_score": float(wisdom_score),
    "signal_label": signal_label,
    "rating": signal_label,
    "sub_scores": {
        "diversity_score": float(diversity_score),
        "independence_score": float(independence_score),
        "decentralization_score": float(decentralization_score),
        "aggregation_score": float(aggregation_score),
    },
    "metrics": {
        "top5_volume_share": float(top5_share),
        "suspicious_volume_share": float(suspicious_volume_share),
        "gini_coefficient": float(gini),
        "network_centralization": float(centralization),
        "modularity": float(modularity),
        "outlier_rate": float(outlier_rate),
    },
    "network": {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "density": float(nx.density(G)),
        "communities": len(communities),
    },
}

with open(PROCESSED_DIR / "wisdom_score_summary.json", "w") as f:
    json.dump(wisdom_data, f, indent=2)
print("Saved: wisdom_score_summary.json")

with open(PROCESSED_DIR / "network_stats.txt", "w") as f:
    f.write("Network Analysis Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Wisdom of Crowds Mechanism Score: {wisdom_score:.1f} / 100\n")
    f.write(f"Signal Label: {signal_label}\n\n")
    f.write("Sub-Scores (Surowiecki Conditions):\n")
    f.write(f"  Diversity Score    : {diversity_score:.1f} / 100\n")
    f.write(f"  Independence Score : {independence_score:.1f} / 100\n")
    f.write(f"  Decentralization   : {decentralization_score:.1f} / 100\n")
    f.write(f"  Aggregation Score  : {aggregation_score:.1f} / 100\n\n")
    f.write("Network Metrics:\n")
    f.write(f"  Nodes: {G.number_of_nodes():,}\n")
    f.write(f"  Edges: {G.number_of_edges():,}\n")
    f.write(f"  Density: {nx.density(G):.4f}\n")
    f.write(f"  Communities: {len(communities)}\n")
    f.write(f"  Modularity: {modularity:.4f}\n\n")
    f.write("Volume Concentration:\n")
    f.write(f"  Top 5 wallets: {top5_share:.1%}\n")
    f.write(f"  Suspicious-group (DBSCAN cluster -1): {suspicious_volume_share:.1%}\n")
    f.write(f"  Gini coefficient: {gini:.3f}\n\n")
    f.write("Network Centralization:\n")
    f.write(f"  Freeman centralization: {centralization:.4f}\n")

print("Saved: network_stats.txt")

print("\n✅ Phase 5 complete. Ready for Phase 6 (Cross-Platform Comparison).")